# 1 Environment Setup

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))
print(f"Using project root: {project_root}\n Please verify path is correct.")
print("Not required if package installed in editable mode via 'pip install -e .'")

Using project root: /home/mbz/Documents/PhD
 Please verify path is correct.
Not required if package installed in editable mode via 'pip install -e .'


In [2]:
from src.config import PipelineConfig
from src.pipeline import SensorPipeline
from src.logging_utils import setup_logging
config = PipelineConfig.from_yaml(Path("config/sensors.yaml"))
print(f"Loaded {len(config.sensors)} sensors:")
for s in config.sensors:
    print(f"  - {s.name} ({s.sensor_type}, altitude: {s.altitude_m}m)")
setup_logging(config.log_dir, log_name="notebook_run")
pipeline = SensorPipeline(config, force=False)

2026-03-03 14:29:17,524 - root - INFO - Logging initialized. Log file: /home/mbz/Documents/PhD/SCD41_DATA/output/logs/notebook_run_20260303_142917.log


Loaded 5 sensors:
  - CO2_SCT1_2M (scd41, altitude: 2m)
  - CO2_30M_A (scd41, altitude: 30m)
  - CO2_30M_B (scd41, altitude: 30m)
  - CO2_30M_C (scd41, altitude: 30m)
  - LGR_30M (lgr_csv, altitude: 30m)


# 2 Process Raw Measurements

### todo:
- collect, process 2025 11, 12 raw data from smear

In [ ]:
# process 2024 9-12
for mo in [9, 10, 11, 12]:
    print(f"\n{'='*30}")
    print(f"Processing Year {2024} | Month {mo:02d}")
    print(f"{'='*30}")
    pipeline.run_all(year=2024, month=mo)

In [ ]:
# process 2025
for mo in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]:
    print(f"\n{'='*30}")
    print(f"Processing Year {2025} | Month {mo:02d}")
    print(f"{'='*30}")
    pipeline.run_all(year=2025, month=mo)

In [ ]:
# process 2026 1, 2
for mo in [1, 2]:
    print(f"\n{'='*30}")
    print(f"Processing Year {2026} | Month {mo:02d}")
    print(f"{'='*30}")
    pipeline.run_all(year=2026, month=mo)

In [ ]:
# process 2024 - 2026
for yr in [2024, 2025, 2026]:
    for mo in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]:
        print(f"\n{'='*30}")
        print(f"Processing Year {yr} | Month {mo:02d}")
        print(f"{'='*30}")
        pipeline.run_all(year=yr, month=mo)

### Check for Duplicates

In [3]:
import pandas as pd
# Check for duplicates in individual CSVs
for sensor in ['CO2_SCT1_2M', 'CO2_30M_A', 'CO2_30M_B', 'CO2_30M_C']:
    df = pd.read_csv(f'output/sensors/{sensor}/csv/{sensor}_10min.csv', parse_dates=['timestamp'])
    dups = df['timestamp'].duplicated().sum()
    print(f"{sensor}: {dups} duplicates")

CO2_SCT1_2M: 0 duplicates
CO2_30M_A: 0 duplicates
CO2_30M_B: 0 duplicates
CO2_30M_C: 0 duplicates


# 3 Merge Sensor Data

## maybe for v0.3.0: Merged output consolidation

> Target: after pipeline output format stabilises (post v0.2.0 per-sensor refactor)

### Planned changes to `output/merged/`

Current structure:

```
merged/
├── csv/          # monthly merged CSVs + total merged CSV
└── parquet/      # monthly merged Parquets (no total)
```

Proposed:

```
merged/
├── csv/
│   ├── SMEAR_EE_CO2_2024_09_merged.csv
│   ├── ...
│   └── SMEAR_EE_CO2_merged.csv          # (exists) total combined
├── parquet/
│   ├── SMEAR_EE_CO2_2024_09_merged.parquet
│   ├── ...
│   └── SMEAR_EE_CO2_merged.parquet      # TODO: total combined parquet
└── uptime/
    └── SMEAR_EE_CO2_uptime.parquet      # TODO: all sensors, all months
```

### Rationale
- **Total combined parquet**: mirrors the existing total CSV; single-read 
  for analysis notebooks and vis package (columnar compression, faster I/O)
- **Combined uptime parquet**: single file with `sensor_id` column; replaces 
  per-sensor CSV reads in `UptimePlots`; enables one-load-filter-plot pattern
- Vis package refactored to consume these two files instead of per-sensor CSVs

### Dependencies
- Requires stable per-sensor directory structure (v0.2.0)
- Vis package update is downstream — only refactor after merged outputs settle

In [4]:
from src.merger import merge_all_sensors

merged = merge_all_sensors(config=config)

print(f"Merged dataset shape: {merged.shape}")
print(f"Date range: {merged.index.min()} to {merged.index.max()}")
print(f"\nColumns: {list(merged.columns)}")

2026-03-03 10:30:26,774 - src.merger - INFO - Merging 5 sensors...
2026-03-03 10:30:26,775 - src.merger - INFO -   Loading CO2_SCT1_2M...
2026-03-03 10:30:26,890 - src.merger - INFO -     66799 rows (2024-09-12 12:20:00 to 2025-12-28 07:00:00)
2026-03-03 10:30:26,891 - src.merger - INFO -   Loading CO2_30M_A...
2026-03-03 10:30:26,938 - src.merger - INFO -     34954 rows (2024-09-19 11:10:00 to 2026-02-03 23:50:00)
2026-03-03 10:30:26,939 - src.merger - INFO -   Loading CO2_30M_B...
2026-03-03 10:30:26,998 - src.merger - INFO -     28902 rows (2024-09-19 11:10:00 to 2026-02-18 14:30:00)
2026-03-03 10:30:26,999 - src.merger - INFO -   Loading CO2_30M_C...
2026-03-03 10:30:27,097 - src.merger - INFO -     61992 rows (2024-10-02 09:30:00 to 2026-02-18 14:30:00)
2026-03-03 10:30:27,098 - src.merger - INFO -   Loading LGR_30M...
2026-03-03 10:30:27,665 - src.merger - INFO -     111053 rows (2024-01-01 00:00:00 to 2026-02-18 14:50:00)
2026-03-03 10:30:27,666 - src.merger - INFO -   Merging o

Merged dataset shape: (111747, 101)
Date range: 2024-01-01 00:00:00 to 2026-02-18 14:50:00

Columns: ['co2_sct1_2m_co2_mean', 'co2_sct1_2m_co2_std', 'co2_sct1_2m_co2_count', 'co2_sct1_2m_temp_mean', 'co2_sct1_2m_temp_std', 'co2_sct1_2m_temp_count', 'co2_sct1_2m_humidity_mean', 'co2_sct1_2m_humidity_std', 'co2_sct1_2m_humidity_count', 'co2_sct1_2m_co2_qc', 'co2_sct1_2m_temp_qc', 'co2_sct1_2m_humidity_qc', 'co2_30m_a_co2_mean', 'co2_30m_a_co2_std', 'co2_30m_a_co2_count', 'co2_30m_a_temp_mean', 'co2_30m_a_temp_std', 'co2_30m_a_temp_count', 'co2_30m_a_humidity_mean', 'co2_30m_a_humidity_std', 'co2_30m_a_humidity_count', 'co2_30m_a_co2_qc', 'co2_30m_a_temp_qc', 'co2_30m_a_humidity_qc', 'co2_30m_b_co2_mean', 'co2_30m_b_co2_std', 'co2_30m_b_co2_count', 'co2_30m_b_temp_mean', 'co2_30m_b_temp_std', 'co2_30m_b_temp_count', 'co2_30m_b_humidity_mean', 'co2_30m_b_humidity_std', 'co2_30m_b_humidity_count', 'co2_30m_b_co2_qc', 'co2_30m_b_temp_qc', 'co2_30m_b_humidity_qc', 'co2_30m_c_co2_mean', 'co2_3

# 4 Validate Data

In [5]:
from src.validation import validate_outputs
validate_outputs()

/home/mbz/Documents/PhD/SCD41_DATA/src/validation/checks.py:446: DtypeWarning: Columns (10,11,12,22,23,24,34,35,36,46,47,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(merged_csv, parse_dates=['timestamp'])
/home/mbz/Documents/PhD/SCD41_DATA/src/validation/checks.py:542: DtypeWarning: Columns (10,11,12,22,23,24,34,35,36,46,47,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(merged_csv, parse_dates=['timestamp'])
2026-03-03 10:30:48,079 - src.validation - INFO -   JSON report: /home/mbz/Documents/PhD/SCD41_DATA/output/validation/validation_report.json
2026-03-03 10:30:48,107 - src.validation - INFO -   Markdown receipt: /home/mbz/Documents/PhD/SCD41_DATA/output/validation/VALIDATION_RECEIPT.md



SMEAR ESTONIA CO2 DATA VALIDATION RECEIPT
Generated: 2026-03-03T10:30:43.740332
Sensors dir:      /home/mbz/Documents/PhD/SCD41_DATA/output/sensors
Merged dir:       /home/mbz/Documents/PhD/SCD41_DATA/output/merged
Year-month range: 2024-01 to 2026-02

1. COLUMN NAME VALIDATION
------------------------------------------------------------
  │ Verifies column names are clean Python identifiers.
  │ Checks for '<lambda>' artifacts from pandas aggregation bugs.
  │ PASS = no problematic column names found..
  │
  │ Status: ✓ ALL PASSED (5/5)
  └──────────────────────────────────────────────────
    ✓ CO2_30M_A/CO2_30M_A_10min.csv: Clean
    ✓ CO2_30M_B/CO2_30M_B_10min.csv: Clean
    ✓ CO2_30M_C/CO2_30M_C_10min.csv: Clean
    ✓ CO2_SCT1_2M/CO2_SCT1_2M_10min.csv: Clean
    ✓ LGR_30M/LGR_30M_10min.csv: Clean

2. FILE INVENTORY
------------------------------------------------------------
  │ Confirms CSV and Parquet files exist for each sensor/month.
  │ PASS = matching CSV and Parquet counts

# 5 Plot

### todo:
 - add options for different time resolutions (hourly, daily)?
 - add options for different date ranges?
 - include temp hue for raw timeseries
 - compound lgr v A,B,C,2m timeseries w/ temp/rh
 - compound lgr v A,B,C,2m uptime? w/ temp/rh
 - refactor validation_plots.py: 
   - inherit single pd read_csv... (try read parquet?; for v0.2.0? --optional... works)

In [3]:
from src.visualization import generate_all_plots
# generate_all_plots(config, show=False, exclude=['scatter','uptime','co2','qc','monthly','comparison','difference','taylor'])
# generate_all_plots(config, show=False, exclude=['scatter', 'co2','uptime','monthly','comparison','difference','taylor'])
# generate_all_plots(config, show=False, exclude=['taylor'])
generate_all_plots(config, show=False, exclude=['scatter','uptime','co2','qc','monthly','comparison','difference'])
# quiet — save only
# Exclude categories: 'uptime', 'qc', 'co2', 'monthly', 'uptime_summary', 'comparison', 'scatter', 'difference', 'taylor', 'scatter_hue', 'timeseries_hue'

2026-03-03 14:29:22,511 - src.visualization - INFO - Generating plots...
2026-03-03 14:29:22,513 - src.visualization - INFO -   Multi-sensor uptime summary
2026-03-03 14:29:23,000 - UptimePlots - INFO - Saved: uptime_summary_all_sensors.png
2026-03-03 14:29:23,002 - src.visualization - INFO -   Loading merged CSV: SMEAR_EE_CO2_merged.csv
/home/mbz/Documents/PhD/SCD41_DATA/src/visualization/__init__.py:233: DtypeWarning: Columns (10,11,12,22,23,24,34,35,36,46,47,48) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_df = _pd.read_csv(merged_csv)
2026-03-03 14:29:23,719 - src.visualization - INFO -   Scatter panel (temp hue)
2026-03-03 14:29:26,319 - ValidationPlots - INFO - Saved: scatter_panel_temp_hue.png
2026-03-03 14:29:26,321 - src.visualization - INFO -   Scatter panel (rh hue)
2026-03-03 14:29:28,860 - ValidationPlots - INFO - Saved: scatter_panel_rh_hue.png
2026-03-03 14:29:28,862 - src.visualization - INFO -   Timeseries (temp hue)
2026-03-03 14: